In [1]:
# RGTransformer 训练器
# 说明：每个阶段都可以独立运行，不依赖其他阶段的执行
%reload_ext autoreload
%autoreload 2

# %% 0. 导入必要的库
import sys
import os

sys.path.append('X:/Workspace/3D-Ocean')

from src.trainer.base import BaseTrainer, BasePrediction
from src.models.SST.RGTransformer import RGTransformer
from src.config.area import Area
from src.config.params import PROJECT_PATH
from src.dataset.OISST import OISSTMonthlyDataset

print("✅ 库导入完成")
print(f"项目根目录: {PROJECT_PATH}")


x:\WorkSpace\3D-Ocean\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
x:\WorkSpace\3D-Ocean\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

✅ 库导入完成
项目根目录: X:/Workspace/3D-Ocean


In [2]:
# %% 全局配置（所有阶段共享）
# 这个cell定义了所有阶段共享的配置参数，每次运行任何阶段前都需要先运行这个cell

area = Area('Global', lon=[-180, 180], lat=[-80, 80], description='全球区域')

# 基础配置参数
resolution = 1
seq_len = 2
offset = 0
use_checkpoint = True

# 计算空间尺寸
width = int(area.width / resolution)
height = int(area.height / resolution)

# 数据集参数
dataset_params = {
    "seq_len": seq_len,
    "offset": offset,
    "resolution": resolution,
}

# 训练器参数（基础配置）
trainer_epochs = 100 # 先定义训练轮数

trainer_params = {
    "epochs": trainer_epochs,
    "batch_size": 16,
    "num_workers": 12,
    "use_wandb": True,
    "use_checkpoint": use_checkpoint,
    "save_top_k": 1,
    "monitor": "val_loss",
    "mode": "min",
}

# 模型参数（基础配置）
rg_transformer_m_params = {
    "width": width,
    "height": height,
    "resolution": resolution,
    "lat_range": area.lat,
    "lon_range": area.lon,
    "seq_len": seq_len,
    "d_model": 512,
    "num_heads": 8,
    "dim_feedforward": 256,
    "dropout": 0.1,
    "recursion_depth": 2,
    "learning_rate": 1e-4,  # 初始学习率
}

# Checkpoint路径（使用项目根目录的绝对路径）
CHECKPOINT_DIR = f'{PROJECT_PATH}/out/checkpoints'
CHECKPOINT_FILE = 'RGTransformer.ckpt'

print("=" * 70)
print("📋 全局配置")
print("=" * 70)
print(f"区域: {area.title} ({area.description})")
print(f"分辨率: {resolution}°")
print(f"序列长度: {seq_len}")
print(f"空间尺寸: {width} x {height}")
print(f"Checkpoint 文件: {CHECKPOINT_FILE}")
print("=" * 70)

📋 全局配置
区域: Global (全球区域)
分辨率: 1°
序列长度: 2
空间尺寸: 160 x 360
Checkpoint 文件: RGTransformer.ckpt


In [3]:
# %% 1. 预训练阶段
# 说明：可以独立运行，从头开始训练模型
# 如果已有checkpoint，可以选择继续训练或重新训练

print("=" * 70)
print("🚀 预训练阶段")
print("=" * 70)

print(f"\n模型: RGTransformer")
print(f"训练轮数: {trainer_params['epochs']}")
print(f"学习率: {rg_transformer_m_params['learning_rate']}")
print(f"Checkpoint: {'启用' if use_checkpoint else '禁用'}")
print("=" * 70 + "\n")

# 创建训练器
pretrain_trainer = BaseTrainer(
    area=area,
    model_class=RGTransformer,
    dataset_class=OISSTMonthlyDataset,# 如果要从已有checkpoint继续，设置此参数
    use_checkpoint=use_checkpoint,
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=rg_transformer_m_params,
)

# 开始训练
pretrain_model = pretrain_trainer.train()

print("\n" + "=" * 70)
print("✅ 预训练完成！")
print("=" * 70)
print(f"最优模型保存在: {CHECKPOINT_DIR}/")
print("=" * 70)

🚀 预训练阶段

模型: RGTransformer
训练轮数: 100
学习率: 0.0001
Checkpoint: 启用

起始时间：1981-09-01


FileNotFoundError: OISST数据文件不存在: X:/Workspace/OISST\sst.mon.mean.1.0deg.nc

In [ ]:
# %% 2. 验证阶段
# 说明：可以独立运行，评估已训练模型的性能
# 需要先有训练好的checkpoint

print("=" * 70)
print("📊 验证阶段")
print("=" * 70)

# 创建评估训练器（不训练，只加载模型进行评估）
eval_trainer = BasePrediction(
    wandb_run_id=pretrain_trainer.trainer_uid,
    area=area,
    model_class=RGTransformer,
    dataset_class=OISSTMonthlyDataset,
    dataset_params=dataset_params,
    model_params=rg_transformer_m_params,
)

# 单个时间点详细预测（可选，绘制图表）
print("\n" + "-" * 70)
print("单个时间点详细预测:")
print("-" * 70)


single_result = eval_trainer.predict(offset=520, plot=True)

print("=" * 70)


In [ ]:
# %% 3. 区域性分析
# 说明：对关键海洋区域进行针对性分析，生成详细的区域统计报告和可视化图表
# 需要先有训练好的checkpoint

from src.analysis.regional import run_regional_analysis

print("=" * 70)
print("🌊 区域性分析阶段")
print("=" * 70)

# 如果还没有创建 eval_trainer，先创建
if 'eval_trainer' not in dir():
    eval_trainer = BasePrediction(
        wandb_run_id=pretrain_trainer.trainer_uid,  # 修改为你的 run_id
        area=area,
        model_class=RGTransformer,
        dataset_class=OISSTMonthlyDataset,
        dataset_params=dataset_params,
        model_params=rg_transformer_m_params,
    )

# 运行完整的区域分析
# 参数说明：
#   - test_offset: 测试时间点
#   - save_dir: 保存目录
#   - detail_regions: 需要详细分析的区域，可选：
#       'nino34'（厄尔尼诺监测区）, 'nino3'（东太平洋暖池）, 
#       'warm_pool'（赤道太平洋暖池）, 'gulf_stream'（墨西哥湾暖流）,
#       'kuroshio'（黑潮）, 'acc'（南大洋西风漂流）, 
#       'north_indian'（北印度洋）, 'north_atlantic_subpolar'（北大西洋副极地）
#   - show_plots: 是否显示图表

analyzer, stats, summary_df = run_regional_analysis(
    predictor=eval_trainer,
    area=area,
    resolution=resolution,
    test_offset=520,
    save_dir='out/sst/regional',
    detail_regions=['nino34', 'gulf_stream', 'kuroshio', 'acc', 'warm_pool'],
    show_plots=True
)

print("\n✅ 区域分析完成！")
